# Notebook 6: Hybrid retrieval (TF-IDF + dense embeddings)

**Goal:** Test an **alternative to pure dense retrieval**: combine **TF-IDF+SVD** and **sentence-transformer** channels with **Reciprocal Rank Fusion (RRF)** — no extra training, parameter-free merge.

**Why:** On synthetic data, sparse lexical similarity often beat dense-only (NB2). Fusing both can stabilize retrieval.

**Evaluation:** Same **item-to-item co-preference** protocol as NB2/NB3 (orders + favorites in test, ≥5 positives per user).

**Text config:** `dish_to_rich_text` with **improved** flags (aligned with NB3 `full_improved`): no recipe, no macro tokens, ratios + ingredients.


In [ ]:
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"
ENV = "kaggle" if os.path.exists("/kaggle/working") else "colab"
BASE = "/kaggle/working" if ENV == "kaggle" else "/content"
REPO_DIR = f"{BASE}/{REPO}"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} ; git pull -q")
os.system("pip install -q sentence-transformers faiss-cpu pandas pyarrow scikit-learn tqdm")
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"ENV={ENV} | {REPO_DIR}")


In [ ]:
import os

ENV = "kaggle" if os.path.exists("/kaggle/working") else "colab" if os.path.exists("/content") else "local"
if ENV == "local":
    SYNTHETIC_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "synthetic")
else:
    SYNTHETIC_DIR = os.path.join(REPO_DIR, "food-recsys-embeddings", "data", "synthetic")
OUTPUT_DIR = "/kaggle/working/processed" if ENV == "kaggle" else SYNTHETIC_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(SYNTHETIC_DIR), SYNTHETIC_DIR
print("SYNTHETIC_DIR", SYNTHETIC_DIR)
print("OUTPUT_DIR", OUTPUT_DIR)


In [ ]:
from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from utils import evaluate_all
import faiss
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)


In [ ]:
dishes = pd.read_parquet(f"{SYNTHETIC_DIR}/dishes.parquet")
test = pd.read_parquet(f"{SYNTHETIC_DIR}/interactions_test.parquet")
print(len(dishes), "dishes |", len(test), "test interactions")


## 1. Texts and index mappings (same row order for dense + TF-IDF)


In [ ]:
texts = []
dish_id_to_idx = {}
idx_to_dish_id = {}
for i, (_, row) in enumerate(dishes.iterrows()):
    t = dish_to_rich_text(
        row.to_dict(),
        tags=row.get("tag_list", []),
        include_recipe=False,
        include_macro_tokens=False,
        include_ratios=True,
        include_ingredients=True,
    )
    texts.append(t)
    did = row["id"]
    dish_id_to_idx[did] = i
    idx_to_dish_id[i] = did
print("N =", len(texts))


## 2. TF-IDF + SVD and dense embeddings


In [ ]:
print("TF-IDF + SVD(256)...")
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X = tfidf.fit_transform(texts)
svd = TruncatedSVD(n_components=256, random_state=42)
tfidf_dense = svd.fit_transform(X).astype(np.float32)
faiss.normalize_L2(tfidf_dense)
index_tfidf = faiss.IndexFlatIP(256)
index_tfidf.add(tfidf_dense)

print("Dense embeddings...")
model = EmbeddingModel()
emb_dense = model.encode(texts)
index_dense = faiss.IndexFlatIP(model.dim)
index_dense.add(emb_dense)
print("tfidf", tfidf_dense.shape, "| dense", emb_dense.shape)


## 3. Evaluators: single-channel vs hybrid RRF


In [ ]:
def evaluate_retrieval_single(index, embeddings, ks=None, min_positives=5):
    if ks is None:
        ks = [5, 10, 20]
    user_positives = (
        test[test["interaction_type"].isin(["order", "favorite"])]
        .groupby("user_id")["dish_id"]
        .apply(set)
        .to_dict()
    )
    results = []
    for _, pos_dishes in user_positives.items():
        pos_dishes = {d for d in pos_dishes if d in dish_id_to_idx}
        if len(pos_dishes) < min_positives:
            continue
        query_dish = list(pos_dishes)[0]
        relevant = pos_dishes - {query_dish}
        qi = dish_id_to_idx[query_dish]
        _, ind = index.search(embeddings[qi : qi + 1], max(ks) + 1)
        recommended = [
            idx_to_dish_id[i]
            for i in ind[0]
            if i >= 0 and idx_to_dish_id.get(i) != query_dish
        ]
        results.append(evaluate_all(recommended, relevant, ks=ks))
    if not results:
        return {}, 0
    return pd.DataFrame(results).mean().to_dict(), len(results)


def rrf_fuse_two_rankings(
    ranked_a: list[int],
    ranked_b: list[int],
    rrf_k: int = 60,
) -> list[int]:
    scores = {}
    for rank, idx in enumerate(ranked_a):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(ranked_b):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)


def evaluate_hybrid_rrf(
    ks=None,
    min_positives=5,
    retrieve_k=50,
    rrf_k=60,
):
    if ks is None:
        ks = [5, 10, 20]
    user_positives = (
        test[test["interaction_type"].isin(["order", "favorite"])]
        .groupby("user_id")["dish_id"]
        .apply(set)
        .to_dict()
    )
    results = []
    cap = max(ks) + 15
    for _, pos_dishes in user_positives.items():
        pos_dishes = {d for d in pos_dishes if d in dish_id_to_idx}
        if len(pos_dishes) < min_positives:
            continue
        query_dish = list(pos_dishes)[0]
        relevant = pos_dishes - {query_dish}
        qi = dish_id_to_idx[query_dish]
        _, id_d = index_dense.search(emb_dense[qi : qi + 1], retrieve_k)
        _, id_t = index_tfidf.search(tfidf_dense[qi : qi + 1], retrieve_k)
        list_d = [int(i) for i in id_d[0] if i >= 0 and int(i) != qi]
        list_t = [int(i) for i in id_t[0] if i >= 0 and int(i) != qi]
        fused = rrf_fuse_two_rankings(list_d, list_t, rrf_k=rrf_k)
        recommended = []
        for row_idx in fused:
            if len(recommended) >= cap:
                break
            did = idx_to_dish_id.get(row_idx)
            if did is not None and did != query_dish:
                recommended.append(did)
        results.append(evaluate_all(recommended, relevant, ks=ks))
    if not results:
        return {}, 0
    return pd.DataFrame(results).mean().to_dict(), len(results)


## 4. Run: dense-only | TF-IDF-only | hybrid RRF


In [ ]:
m_dense, n1 = evaluate_retrieval_single(index_dense, emb_dense)
m_tfidf, n2 = evaluate_retrieval_single(index_tfidf, tfidf_dense)
m_hybrid, n3 = evaluate_hybrid_rrf()
assert n1 == n2 == n3, (n1, n2, n3)
print(f"n_users evaluated = {n1}")

rows = {
    "Dense only (improved text)": m_dense,
    "TF-IDF+SVD only": m_tfidf,
    "Hybrid RRF (dense + TF-IDF)": m_hybrid,
}
comp = pd.DataFrame(rows).T
print("\n=== Comparison (same protocol) ===")
cols = [c for c in ["P@5", "P@10", "NDCG@10", "MRR", "HR@10"] if c in comp.columns]
print(comp[cols].round(4).to_string())
best_single = comp["P@10"].drop(labels="Hybrid RRF (dense + TF-IDF)", errors="ignore").max()
delta = comp.loc["Hybrid RRF (dense + TF-IDF)", "P@10"] - best_single
print(f"\nHybrid P@10 minus best single-channel P@10: {delta:+.4f}")


## 5. Save metrics


In [ ]:
import json

out = {
    "dense_improved_text": m_dense,
    "tfidf_svd": m_tfidf,
    "hybrid_rrf": m_hybrid,
    "n_users": n1,
}
path = os.path.join(OUTPUT_DIR, "hybrid_results.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)
print("Saved", path)
